# SeaScan — complete Sentinel-1 dataset training

This notebook trains the backend-compatible SeaScan U-Net without using storage on your Mac. It processes the official Zenodo archives sequentially: Parts I and II are used for scene-level training/validation, while Part III remains untouched until the final evaluation.

Use a GPU runtime with at least **90 GiB free ephemeral disk**. The final checkpoint and test report are saved to Google Drive. Temporary raw archives are deleted from /content after each category is converted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get update -qq
!apt-get install -y -qq aria2 p7zip-full
!test -d /content/SeaScan/.git && git clone https://github.com/Prachi2424/SeaScan.git /content/SeaScan || git -C /content/SeaScan pull --ff-only
%cd /content/SeaScan
!pip install -q -r backend/requirements.txt

In [ ]:
import shutil
import torch

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT ENABLED')
print('Free disk GiB:', round(shutil.disk_usage('/content').free / 2**30, 1))
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU before continuing.'
assert shutil.disk_usage('/content').free / 2**30 >= 90, 'Choose a runtime with at least 90 GiB free disk.'

## Run the complete pipeline

The download and preprocessing stage is long. The script also supports prepare, train, and test stages separately if the runtime is interrupted. Do not inspect Part III repeatedly while tuning the model.

In [ ]:
WORK_DIR = '/content/seascan-full-dataset'
CHECKPOINT = '/content/drive/MyDrive/SeaScan/checkpoints/seascan_unet_full.pt'
EPOCHS = 20
BATCH_SIZE = 16

!python training/full_dataset_cloud.py all --work-dir {WORK_DIR} --checkpoint {CHECKPOINT} --epochs {EPOCHS} --batch-size {BATCH_SIZE}

## Outputs

After completion, Google Drive contains seascan_unet_full.pt and seascan_unet_full.test-metrics.json. Download only these small files to integrate the trained model into SeaScan.